### Link Grabber


In [1]:
from selenium import webdriver
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager

from bs4 import BeautifulSoup
from pathlib import Path
from tqdm import tqdm

import pandas as pd
import time
import re

import tiktok_lib as tt

options = Options()
service = Service(ChromeDriverManager().install())


c:\Users\vi\Documents\GitHub\Video_detection_model-WIP-\scripts\scraper\tiktok_lib.py:25: SyntaxWarning: invalid escape sequence '\.'
  url_regex = '(?<=\.com/)(.+?)(?=\?|$)'


If pyktok does not operate as expected, you may find it helpful to run the 'specify_browser' function. 'specify_browser' takes as its sole argument a string representing a browser installed on your system, e.g. "chrome," "firefox," "edge," etc.


In [2]:
def grab_tiktok_links(
        url: str = 'https://www.tiktok.com/tag/ai',
        a_class: str = 'AMetaCaptionLine',
        wait_time: int = 1,
        max_tries: int = 3
        ) -> list[str]:
    """
    Args:
        url: Tiktok hashtag url
        a_class: <a> container class for lookup
        wait_time: Seconds to pause between checks. Increase if poor internet
        max_tries: Number of failed scrolls before exiting

    Returns:
        links: Unique urls. Size not match goal.
    """
    driver = webdriver.Chrome(service=service, options=options)
    driver.get(url)

    wait = WebDriverWait(driver, timeout=5)
    html = driver.find_element(By.TAG_NAME,"html")
    links = set()
    
    a_selector = f"a[class*='{a_class}']"
    count = 0
    wait.until(
        lambda d: len(d.find_elements(By.CSS_SELECTOR, a_selector)) != 0
    )
    prev_elements = []
    try:
        while count <= max_tries:
            elements = driver.find_elements(By.CSS_SELECTOR, a_selector)

            for el in elements:
                href = el.get_attribute("href")
                if href:
                    links.add(href.split("?")[0])

            html.send_keys(Keys.END)
            
            time.sleep(wait_time)

            if elements == prev_elements:
                count += 1
            else:
                count = 0
            prev_elements = elements

    finally:
        driver.quit()

    print(f"Successfully grabbed {len(links)} urls.")
    
    return list(links)


In [3]:
def save_links(
        links: list,
        path: Path = Path("csvs"),
        filename: str = "urls.csv",
        replace: bool = False
        ) -> None:
    """
    Args:
        path: Defaults to "csvs" folder in same dir
        replace: Whether to overwrite existing files sharing the filename. False will increment a counter suffix.
    """
    df = pd.DataFrame(links, columns=["url"])

    path.mkdir(parents=True, exist_ok=True)

    filepath = path / filename

    if not replace:
        stem = Path(filename).stem
        suffix = Path(filename).suffix

        count = 0

        while filepath.exists():
            filepath = path / f"{stem}_{count}{suffix}"
            count += 1

    df.to_csv(filepath, index=False)

In [ ]:
def save_video_from_url(
        url: str,
        path: Path = Path("data")
        ) -> tuple[str, str]:
    """
    Args:
        url: TikTok video url
        path: Defaults to "data" folder in same dir
    Returns:
        (username, video_id) string tuple
    """
    path.mkdir(parents=True, exist_ok=True)

    # Example URL: https://www.tiktok.com/@username/video/7589040432898657550

    username_match = re.search(r'@([^/]+)', url)
    username = username_match.group(1) if username_match else None

    video_id_match = re.search(r'/video/(\d+)', url)
    video_id = video_id_match.group(1) if video_id_match else None

    tt.save_tiktok(url, True,
                   video_fn = path / f"@{username}_{video_id}.mp4",
                   metadata_fn = path / 'metadata.csv'
                   )
    return username, video_id


def save_video_batch(
        links: list[str] = [],
        link_folder: Path = Path('csvs'),
        start: int = 0,
        goal: int = 10,
        wait: int = 0,
        path: Path = Path("data")
        ) -> None:
    
    """
    Args:
        links: List of video URLs. If none given, will search for them
        start: From what index to begin
        goal: How many videos to download before stopping
        wait: Seconds to wait between downloads
        path: Defaults to "data" folder in same dir
    """

    if not links:
        csv_files = list(link_folder.glob('*.csv'))
        
        if csv_files:
            dfs = [pd.read_csv(file) for file in csv_files]
            combined_df = pd.concat(dfs, ignore_index=True)
            
            links = combined_df['url'].tolist()
            print(f"Loaded {len(links)} URLs from {len(csv_files)} CSV files")
        else:
            print(f"No CSV files found in {link_folder}")

    if not links:
        print("No links found, and none could be imported.")
        return
    if start > len(links):
        print(f"Start index {start} is Out of range.")
        return
    end = min(len(links), start + goal)

    for i in tqdm(range(start, end), 
              desc="Downloading videos",
              unit="video"):
        save_video_from_url(links[i], path)
        time.sleep(wait)
    print(f"Videos saved to {path}.")


In [ ]:
res = grab_tiktok_links()
save_links(res)

In [3]:
save_video_batch(goal = 3)

Loaded 153 URLs from 1 CSV files


Saved video
 https://www.tiktok.com/@aiflaivors/video/7594285007565311262 
to
 c:\Users\vi\Documents\GitHub\Video_detection_model-WIP-\scripts\scraper
Saved metadata for video
 https://www.tiktok.com/@aiflaivors/video/7594285007565311262 
to
 c:\Users\vi\Documents\GitHub\Video_detection_model-WIP-\scripts\scraper


Saved video
 https://www.tiktok.com/@talkingthingsss/video/7602399043918007574 
to
 c:\Users\vi\Documents\GitHub\Video_detection_model-WIP-\scripts\scraper
Saved metadata for video
 https://www.tiktok.com/@talkingthingsss/video/7602399043918007574 
to
 c:\Users\vi\Documents\GitHub\Video_detection_model-WIP-\scripts\scraper


Saved video
 https://www.tiktok.com/@theboldszn/video/7601236351677222147 
to
 c:\Users\vi\Documents\GitHub\Video_detection_model-WIP-\scripts\scraper
Saved metadata for video
 https://www.tiktok.com/@theboldszn/video/7601236351677222147 
to
 c:\Users\vi\Documents\GitHub\Video_detection_model-WIP-\scripts\scraper
Videos saved to data.


In [26]:
%pip install playwright

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
